# Diagnóstico de churn RavenStack — apêndice técnico (CRISP-DM)

Este notebook **explica** o pipeline; ele não reimplementa nada (DRY): toda
conta vem do pacote `churn_diag`, o mesmo que gera `outputs/` e que os testes
verificam. O relatório para o CEO é o `RELATORIO.md`.

| Fase CRISP-DM | Seção | Pergunta |
|---|---|---|
| 1. Entendimento do negócio | §1 | Que tipo de pergunta o CEO está fazendo? |
| 2. Entendimento dos dados | §2 | Dá para confiar nos dados? |
| 3. Preparação | §3 | Qual a unidade de análise correta? |
| 4. Modelagem | §4 | O churn subiu? Onde? O que prevê? |
| 5. Avaliação | §5 | O que sobrevive à correção e à validação fora do tempo? |
| 6. Implantação | §6 | O que o CS faz amanhã e como monitorar? |

> Stack: Python 3.14 + Polars (scikit-learn só recebe arrays NumPy).

> **Kernel:** use o do projeto — *Python 3.14 (churn-diag)*. Ele é o
> ambiente travado pelo `uv.lock`, com Polars e o pacote `churn_diag`
> instalados. Para criá-lo/registrá-lo, em `solution/`:
>
> ```bash
> uv sync
> uv run python -m ipykernel install --user --name churn-diag \
>     --display-name "Python 3.14 (churn-diag)"
> ```
>
> Ou rode o Jupyter já dentro do ambiente: `uv run jupyter lab`.
> A célula abaixo confere o kernel e falha com instrução em vez de um
> `ModuleNotFoundError` seco.

In [1]:
# Guarda de ambiente: kernel certo ou mensagem clara.
import sys
from pathlib import Path

_src = Path.cwd().parent / "src"  # notebooks/ -> solution/src
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))  # permite rodar com o pacote não instalado

try:
    import polars as pl  # noqa: F401
    import churn_diag  # noqa: F401
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"{exc.name} não está neste kernel ({sys.executable}).\n"
        "Selecione o kernel 'Python 3.14 (churn-diag)'. Para criá-lo, em solution/:\n"
        "  uv sync\n"
        "  uv run python -m ipykernel install --user --name churn-diag "
        "--display-name 'Python 3.14 (churn-diag)'\n"
        "Ou rode: uv run jupyter lab"
    ) from exc

print("kernel ok:", sys.executable)

kernel ok: /Volumes/Backup/Projects/src/challengers/ai-master-challenge/submissions/geoffrey-porto/solution/.venv/bin/python3


In [2]:
import sys

import polars as pl

from churn_diag.config import CS_TOP_N, resolve_data_dir
from churn_diag.loader import load_tables
from churn_diag.metrics import exposure_panel, hazard_table, monthly_churn, standardized_ratio
from churn_diag.pipeline import analyse
from churn_diag.quality import cohort_churn_profile, quality_report

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)
print(sys.version.split()[0], "| polars", pl.__version__)
t = load_tables(resolve_data_dir())
t.row_counts()

3.14.0 | polars 1.44.2


{'accounts': 500,
 'subscriptions': 5000,
 'feature_usage': 25000,
 'support_tickets': 2000,
 'churn_events': 600}

## 1. Entendimento do negócio — três perguntas diferentes

O CEO faz, na mesma frase, três perguntas de **tipos** diferentes. Misturá-las
é o erro mais comum neste desafio (usar correlação descritiva para justificar
ação causal):

| Pergunta do CEO | Tipo | Objeto formal | Onde responde |
|---|---|---|---|
| "O que está acontecendo?" | Descrição | $P(Y \mid \text{segmento})$ | §4.1–4.3 |
| "Quem está em risco?" | Predição | $\hat P(Y=1 \mid X=x)$ | §4.5 |
| "O que devemos fazer?" | Causal | $E[Y \mid do(A=a)] - E[Y \mid do(A=a')]$ | §6 (desenho de experimento) |

Por isso cada achado do registro carrega um rótulo (`descricao`, `predicao`,
`hipotese_causal`) — e o princípio P-004 da constituição testa isso.

## 2. Entendimento dos dados — auditoria antes de qualquer gráfico

Primeira coisa que fiz foi checar se as datas contam uma história coerente. Não
contam:

In [3]:
q = quality_report(t)
{k: q[k] for k in ("timeline", "churn_definitions", "reason_vs_feedback", "usage_id_collisions")}

{'timeline': {'usage_before_sub_start': 19142,
  'usage_before_sub_start_pct': 76.6,
  'usage_after_sub_end': 290,
  'usage_before_signup_pct': 52.8,
  'tickets_before_signup': 1077,
  'tickets_before_signup_pct': 53.9,
  'churn_events_before_first_sub': 53,
  'subs_before_signup': 0,
  'subs_end_before_start': 0},
 'churn_definitions': {'accounts': 500,
  'flag_account': 110,
  'has_churn_event': 352,
  'has_ended_subscription': 312,
  'all_three_agree_churn': 50,
  'all_three_agree_active': 50,
  'disagree': 400,
  'event_but_flag_false': 277,
  'flag_true_without_event': 35},
 'reason_vs_feedback': {'chi2': 3.83,
  'p_value': 0.955,
  'cramers_v': 0.065,
  'reason_share_max_pct': 19.0,
  'reason_share_min_pct': 15.2,
  'feedback_missing_pct': 24.7},
 'usage_id_collisions': 21}

**Leitura:**

- 77% do uso acontece **antes** do início da assinatura a que está ligado, e 54%
  dos tickets antes do cadastro da conta. Janelas "30/60/90 dias antes do churn"
  com esses dados (o que o guia de referência sugere) seriam ruído com cara de
  insight — **decisão: não construí essas features**.
- As três definições de churn do dataset (flag da conta, eventos de churn,
  assinatura encerrada) discordam em 80% das contas. A única linha do tempo
  internamente consistente é a da assinatura (`end_date ≥ start_date` em 100%).
- 21 `usage_id` se repetem com conteúdo diferente: colisão de IDs curtos (6 hex
  para 25 mil linhas ≈ o esperado pelo paradoxo do aniversário). Deduplicar por ID
  apagaria eventos reais.
- O motivo de saída e o comentário do cliente são independentes (V de Cramér ≈ 0).

E um teste de sanidade que muda a conversa — a fração de assinaturas que "já
saíram" por coorte de início:

In [4]:
cohort_churn_profile(t)

cohort,subs,ever_ended_pct,days_observed
date,u32,f64,f64
2023-01-01,31,19.4,670.0
2023-04-01,106,10.4,592.0
2023-07-01,203,15.8,503.0
2023-10-01,326,11.0,407.0
2024-01-01,472,9.3,316.0
2024-04-01,686,8.6,227.0
2024-07-01,1107,8.1,134.0
2024-10-01,2069,10.1,38.0


Em SaaS real, coortes mais antigas acumulam **mais** churn. Aqui, a coorte do
4º tri/2024, observada por 38 dias, já perdeu 10,1% — praticamente o mesmo que a
do 4º tri/2023, observada por 407 dias (11%). Com o risco de referência
(~1%/mês), a de 38 dias deveria ter perdido ~1%. Duas explicações produzem esse
padrão: (a) um colapso real do início de vida nas coortes recentes; (b) um
`end_date` atribuído por processo (≈10% de cada coorte recebe uma data de fim
dentro da janela). Os dados sozinhos não separam as duas — vai para o relatório
como alerta e como ação 0 (auditoria de 20 cancelamentos no billing).

## 3. Preparação — a unidade de análise

Unidade = **assinatura × mês** (tempo discreto). Duas visões do mesmo painel:

- `active_at_start` → taxa mensal clássica de SaaS (denominador = base no 1º dia);
- toda linha ativa no mês → risco por idade (inclui quem entra e sai no mesmo mês).

> **Onde eu (IA) errei primeiro:** na exploração inicial usei só "ativa no 1º dia"
> também para o risco por idade. Isso descartava as assinaturas que começam e
> terminam no mesmo mês — justamente o churn de início de vida. O ajuste mudou a
> razão observado/esperado do 4º tri de 1,95× para 2,82× e passou a contar as 486
> saídas (antes eram 341).

In [5]:
panel = exposure_panel(t.subscriptions)
print(panel.height, "linhas assinatura×mês |", int(panel["ended"].sum()), "saídas (todas as 486)")
panel.select("subscription_id", "month", "age_days", "age_bucket", "active_at_start", "ended", "paid_mrr", "period").head()

29192 linhas assinatura×mês | 486 saídas (todas as 486)


subscription_id,month,age_days,age_bucket,active_at_start,ended,paid_mrr,period
str,date,i64,str,bool,bool,i64,str
"""S-18356d""",2023-01-01,0,"""0-30d""",false,false,3582,"""pre"""
"""S-50aaaa""",2023-01-01,0,"""0-30d""",false,false,931,"""pre"""
"""S-92f228""",2023-01-01,0,"""0-30d""",false,false,171,"""pre"""
"""S-0e4b0c""",2023-02-01,0,"""0-30d""",false,false,1176,"""pre"""
"""S-11066a""",2023-02-01,0,"""0-30d""",false,false,931,"""pre"""


## 4. Modelagem

### 4.1 Contagem vs taxa

In [6]:
m = monthly_churn(panel).filter(pl.col("month") >= pl.date(2024, 1, 1))
m.select("month", "active_subs", "ended_all", "sub_churn_rate", "mrr_churn_rate", "period")

month,active_subs,ended_all,sub_churn_rate,mrr_churn_rate,period
date,u32,u32,f64,f64,str
2024-01-01,656,11,0.01372,0.00984,"""reference"""
2024-02-01,776,7,0.00902,0.00931,"""reference"""
2024-03-01,916,9,0.00983,0.01037,"""reference"""
2024-04-01,1105,12,0.01086,0.00644,"""reference"""
2024-05-01,1285,12,0.00934,0.00864,"""reference"""
2024-06-01,1520,13,0.00724,0.00502,"""reference"""
2024-07-01,1752,17,0.0097,0.00725,"""reference"""
2024-08-01,2068,28,0.01161,0.00856,"""reference"""
2024-09-01,2380,35,0.01176,0.00912,"""reference"""


### 4.2 Risco por idade e padronização de mix (Simpson aplicado ao churn)

In [7]:
hazard_table(panel.filter(pl.col("period") != "pre"), ["period", "age_bucket"]).select(
    "period", "age_bucket", "exposures", "events", "hazard", "mrr_hazard")

period,age_bucket,exposures,events,hazard,mrr_hazard
str,str,u32,u32,f64,f64
"""reference""","""0-30d""",4086,39,0.00954,0.0095
"""reference""","""180-365d""",3357,24,0.00715,0.0056
"""reference""","""30-90d""",3147,40,0.01271,0.0089
"""reference""","""365d+""",748,7,0.00936,0.00296
"""reference""","""90-180d""",3297,34,0.01031,0.00897
"""target""","""0-30d""",3536,203,0.05741,0.06167
"""target""","""180-365d""",2595,29,0.01118,0.02104
"""target""","""30-90d""",2226,55,0.02471,0.02955
"""target""","""365d+""",1170,10,0.00855,0.00418


In [8]:
standardized_ratio(panel)

{'observed': 324,
 'expected': 115.0,
 'ratio': 2.82,
 'p_value': 3.8197905212201677e-57,
 'mrr_observed': 871812.0,
 'mrr_expected': 209405.0,
 'mrr_ratio': 4.16}

Se a alta fosse só mudança de mix (mais assinaturas jovens, que naturalmente
saem mais), a razão ficaria perto de 1. Deu 2,82× em assinaturas e 4,16× em MRR —
o risco **por idade** piorou, e piorou nas faixas jovens.

### 4.3 Resultado completo do pipeline

In [9]:
res = analyse(t, CS_TOP_N)
res.tables["findings"].select("id", "claim_type", "title", "effect_name", "effect", "p_value", "p_holm", "significant")

id,claim_type,title,effect_name,effect,p_value,p_holm,significant
str,str,str,str,f64,f64,f64,bool
"""H1""","""descricao""","""O churn do 4º tri/2024 subiu a…","""observado/esperado""",2.82,3.8200e-57,4.5800e-56,true
"""H2""","""descricao""","""A alta está concentrada em ass…","""RR jovem ÷ RR madura""",3.2973,3.1100e-8,3.4200e-7,true
"""H3""","""descricao""","""A satisfação (CSAT) não distin…","""AUC (0,5 = acaso)""",0.5244,0.358,1.0,false
"""H4""","""descricao""","""O uso não cresceu: total estáv…","""variação do uso por assinatura…",-0.832,0.88,1.0,false
"""H5""","""predicao""","""Uso, suporte e satisfação não …","""ROC-AUC fora do tempo""",0.5262,0.385,1.0,false
"""H6""","""descricao""","""Quem declarou 'suporte' como m…","""AUC (0,5 = acaso)""",0.5031,0.93,1.0,false
"""H7""","""descricao""","""O motivo codificado e o coment…","""V de Cramér (0 = independentes…",0.0651,0.955,1.0,false
"""H8""","""hipotese_causal""","""O salto de novas assinaturas n…","""rho de Spearman""",0.6923,0.0126,0.126,false
"""H9""","""descricao""","""Indústria não diferencia o chu…","""V de Cramér""",0.0278,0.53,1.0,false


### 4.4 Invariância (ICP em versão enxuta)

A ideia do ICP (Peters, Bühlmann & Meinshausen): uma relação candidata a causal
é **estável entre ambientes**. Testei a relação principal — idade → risco — em
13 perfis de cliente (indústria, plano, canal) e, por decisão do dono do produto
enquanto a pergunta Q-001 não é respondida, também **antes/depois da quebra de
set–out/2024** (ASM-006, risco alto). Com ~500 contas, um IRM completo não teria
poder; a checagem de estabilidade responde o que importa.

In [10]:
res.tables["invariance"]

env_type,env,exposures,hazard_young,hazard_mature,hr,ci_low,ci_high,direction,ci_excludes_1,stable_in_type
str,str,i64,f64,f64,f64,f64,f64,str,bool,bool
"""industry""","""Cybersecurity""",5030,0.02802,0.01113,2.517,1.624,3.9,"""+""",true,true
"""industry""","""DevTools""",6234,0.02552,0.0132,1.934,1.336,2.799,"""+""",true,true
"""industry""","""EdTech""",3937,0.02354,0.00827,2.845,1.603,5.049,"""+""",true,true
"""industry""","""FinTech""",5923,0.02501,0.00723,3.46,2.152,5.563,"""+""",true,true
"""industry""","""HealthTech""",5318,0.02734,0.00812,3.365,2.104,5.383,"""+""",true,true
"""plan_tier""","""Basic""",8648,0.02364,0.0111,2.129,1.519,2.986,"""+""",true,true
"""plan_tier""","""Enterprise""",8989,0.02617,0.01058,2.472,1.768,3.457,"""+""",true,true
"""plan_tier""","""Pro""",8805,0.02789,0.00755,3.694,2.53,5.394,"""+""",true,true
"""referral_source""","""ads""",5599,0.02713,0.00757,3.586,2.233,5.759,"""+""",true,true


Estável nos 13 perfis de cliente — mas **não no tempo**: antes da quebra a razão
era 1,2× (intervalo 0,90–1,72, sem diferença) e depois 4,1× (3,14–5,36).
"Assinatura nova sai mais" é um **regime novo**, não um traço permanente; a causa
está no evento de set–out/2024 (Q-001). Foi a decisão humana de tratar a quebra
como ambiente que revelou isso. **Ressalva:** invariância não distingue mecanismo
real de defeito de registro — por isso a ação 0 continua antes de qualquer
investimento.

### 4.5 Validação fora do tempo (ERM vs. o simples)

In [11]:
res.tables["oot_validation"]

scorer,roc_auc,pr_auc,lift_at_10,mrr_recall_at_10,p_value,roc_auc_in_sample,n_test,base_rate
str,f64,f64,f64,f64,f64,f64,i64,f64
"""idade_da_assinatura""",0.5858,0.0502,1.25,0.177,0.003409,0.646,2330,0.0412
"""perda_esperada_mrr""",0.5602,0.0536,1.354,0.4896,0.045548,0.5367,2330,0.0412
"""so_mrr""",0.5193,0.0558,1.562,0.5825,0.52134,0.4772,2330,0.0412
"""logistica_todas_tabelas""",0.5334,0.052,1.667,0.0789,0.26686,0.754,2330,0.0412
"""gbm_todas_tabelas""",0.5262,0.0472,1.25,0.1033,0.384711,0.9994,2330,0.0412


O GBM com todas as tabelas atinge ROC 1,00 no treino e 0,52 fora dele: é o
retrato da **subespecificação** — com variáveis sem sinal, o modelo escolhe um
atalho qualquer e decora. Só a idade mantém sinal (0,59, p = 0,003).

Por isso **não** usei SMOTE-NC (sobreamostrar ruído não cria sinal), nem Causal
Forest / PSM (sem variação exógena; um "tratamento" como escalação não tem
associação alguma com o churn — não há o que casar).

## 5. Avaliação — o que sobrevive

In [12]:
{k: res.report[k] for k in ("n_hypotheses", "n_significant", "q4_ratio_x", "rr_young_x",
    "rr_mature_x", "csat_auc", "usage_change_per_sub_pct", "oot_age_roc", "oot_gbm_roc", "h8_p_holm")}

{'n_hypotheses': 12,
 'n_significant': 2,
 'q4_ratio_x': 2.82,
 'rr_young_x': 4.1,
 'rr_mature_x': 1.24,
 'csat_auc': 0.52,
 'usage_change_per_sub_pct': -83.2,
 'oot_age_roc': 0.59,
 'oot_gbm_roc': 0.53,
 'h8_p_holm': 0.13}

## 6. Implantação — o que o CS faz amanhã

Score = perda esperada de MRR em 90 dias (risco mensal por idade, envelhecendo
a assinatura mês a mês, × MRR pago).

In [13]:
res.tables["cs_priority_accounts"].head(10).select("rank", "account_id", "account_name", "active_paid_mrr", "young_subs", "expected_loss_90d", "acao_sugerida")

rank,account_id,account_name,active_paid_mrr,young_subs,expected_loss_90d,acao_sugerida
u32,str,str,i64,u32,f64,str
1,"""A-18793f""","""Company_488""",75776,8,8038.0,"""CSM sênior: call de ativação e…"
2,"""A-d4e0d4""","""Company_403""",114777,9,7781.0,"""CSM sênior: call de ativação e…"
3,"""A-5c046d""","""Company_130""",83886,5,6346.0,"""CSM sênior: call de ativação e…"
4,"""A-4814a3""","""Company_337""",70644,7,5028.0,"""CSM sênior: call de ativação e…"
5,"""A-9174e0""","""Company_73""",46613,12,4945.0,"""CSM sênior: call de ativação e…"
6,"""A-09316c""","""Company_341""",70000,7,4899.0,"""CSM sênior: call de ativação e…"
7,"""A-5b1bcd""","""Company_166""",131911,3,4427.0,"""CSM sênior: call de ativação e…"
8,"""A-56962b""","""Company_177""",54605,1,4242.0,"""CSM sênior: call de ativação e…"
9,"""A-c70870""","""Company_253""",39614,4,4177.0,"""CSM sênior: call de ativação e…"


In [14]:
{k: res.report[k] for k in ("excess_mrr_per_month_k", "recovery_25_mrr_month_k", "recovery_50_mrr_month_k",
    "recovery_75_mrr_month_k", "ab_p0_pct", "ab_p1_pct", "ab_n_per_arm", "ab_weeks_to_enroll")}

{'excess_mrr_per_month_k': 221.0,
 'recovery_25_mrr_month_k': 55.0,
 'recovery_50_mrr_month_k': 110.0,
 'recovery_75_mrr_month_k': 166.0,
 'ab_p0_pct': 5.75,
 'ab_p1_pct': 2.87,
 'ab_n_per_arm': 783,
 'ab_weeks_to_enroll': 11.6}

---

## 7. Validação das features da referência (incremento 2)

O candidato trouxe um documento próprio de engenharia de features (20 features,
janelas de 90 dias) com a sua triagem: ROC 0,604 e precisão média 0,144 num
painel **conta × snapshot mensal**, rótulo = evento de churn não-reativação em
30 dias. Três perguntas ficaram abertas na comparação — e foram medidas.

In [15]:
from churn_diag.account_panel import attach_full_history_rates, build_account_panel, split_train_test
from churn_diag.features import RATE_FEATURES
from churn_diag.screening import age_signal_comparison, published_reference_metrics, reference_replication, univariate_screening

acc = attach_full_history_rates(t, build_account_panel(t))
print("replicação:", reference_replication(acc))
print("publicado :", published_reference_metrics())

replicação: {'train_rows': 3392, 'test_rows': 854, 'train_positive_rate': 0.0693, 'test_positive_rate': 0.1066, 'test_roc_auc': 0.5716, 'test_average_precision': 0.1448}
publicado : {'train_rows': 3392, 'test_rows': 854, 'test_roc_auc': 0.604, 'test_average_precision': 0.144}


O recorte bate linha a linha (3.392 treino / 854 teste, mesmas taxas de evento)
e a precisão média reproduz. O ROC fica ~0,03 abaixo porque a replicação roda
sem cinco features: as três sem data (quarentena) e as duas de tendência e
recência de uso (linha do tempo quebrada).

### 7.1 As três taxas têm sinal?

In [16]:
_, acc_test = split_train_test(acc)
cols = [f"{r}_90d" for r in RATE_FEATURES] + [f"{r}_all" for r in RATE_FEATURES]
univariate_screening(acc_test, cols).select("feature", "n", "auc", "p_holm", "direction", "significant")

feature,n,auc,p_holm,direction,significant
str,i64,f64,f64,str,bool
"""errors_per_100_uses_all""",854,0.5285,1.0,"""maior = mais risco""",false
"""escalation_rate_90d""",356,0.4736,1.0,"""menor = mais risco""",false
"""errors_per_100_uses_90d""",847,0.5251,1.0,"""maior = mais risco""",false
"""escalation_rate_all""",840,0.4815,1.0,"""menor = mais risco""",false
"""satisfaction_missing_share_90d""",356,0.4996,1.0,"""menor = mais risco""",false
"""satisfaction_missing_share_all""",840,0.5002,1.0,"""maior = mais risco""",false


Nenhuma separa quem sai de quem fica — e isso vale também sobre todo o
histórico, então o problema não é só a janela de 90 dias.

### 7.2 Idade da conta e idade da assinatura são o mesmo sinal?

In [17]:
age_signal_comparison(acc).select("modelo", "roc_auc", "spearman_tenure_vs_sub_age", "auc_tenure_days", "auc_min_sub_age_days", "ganho_ao_juntar", "veredito")

modelo,roc_auc,spearman_tenure_vs_sub_age,auc_tenure_days,auc_min_sub_age_days,ganho_ao_juntar,veredito
str,f64,f64,f64,f64,f64,str
"""só idade da conta""",0.6794,0.322,0.3397,0.3902,-0.0318,"""sinais distintos: as idades nã…"
"""só idade da assinatura""",0.5195,0.322,0.3397,0.3902,-0.0318,"""sinais distintos: as idades nã…"
"""as duas""",0.6476,0.322,0.3397,0.3902,-0.0318,"""sinais distintos: as idades nã…"


**Sinais distintos.** Correlação fraca (0,32); no painel por conta a idade da
conta prevê bem melhor que a idade da assinatura, e juntar as duas piora. O
`tenure_days` da referência não é o mesmo efeito que a idade da assinatura deste
diagnóstico.

> **Ressalva:** empilhando snapshots mensais, `tenure_days` também carrega a
> coorte de entrada. Como o rótulo dispara no 4º tri/2024, parte do poder
> preditivo pode ser o mesmo regime novo da pergunta Q-001.

### 7.3 Quanto custou tirar as flags sem data?

`upgrade_flag`, `downgrade_flag` e `auto_renew_flag` não têm carimbo de tempo:
não dá para provar que descrevem o cliente antes do corte (princípio P-009).
Saíram de todas as matrizes — e os modelos fora do tempo **não pioraram**
(logística 0,53; GBM 0,52 → 0,53).

---

## 8. Plano de monitoramento

| Sinal | Métrica | Gatilho | Ação |
|---|---|---|---|
| Quebra na taxa | churn de MRR mensal vs. limite de controle (média + 3σ) | 1 mês acima | investigar a causa de negócio primeiro |
| Churn de início de vida | risco dos primeiros 30 dias | > 2× a referência | revisar onboarding e o que Vendas mudou |
| Qualidade de dados | % de eventos fora da janela da assinatura | > 1% | bloquear o score até corrigir |
| Estabilidade do score | ROC fora do tempo por trimestre | < 0,55 | recalibrar a tabela de risco |

Modelos temporais pesados (TimesFM) ficam para quando houver série longa e
dados confiáveis — aqui, o controle estatístico simples já detectou a quebra.